# VQA CAA application evaluation

## Gemma 3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### defs

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import json
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm
import kagglehub
from pathlib import Path
from os.path import join

In [ ]:
model_id = "google/gemma-3-4b-it"

In [ ]:
def load_model():
  processor = AutoProcessor.from_pretrained(model_id)
  model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.bfloat16
  ).eval()
  return processor, model

In [ ]:
proc, mod = load_model()

In [ ]:
mod

## DS

### prepare steering vectors

done in *VQA_CAA_create.ipynb*

### src data

In [ ]:
path = kagglehub.dataset_download("timoboz/clevr-dataset")
print(path)
img_dir = join(path, 'CLEVR_v1.0/images/val')

### evaluation ds

In [ ]:
class CLEVRCAAEvaluationDataset(Dataset):
    def __init__(self, jsonl_path, image_dir):
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().lstrip('\ufeff')

        decoder = json.JSONDecoder()
        idx = 0
        while idx < len(content):
            next_brace = content.find('{', idx)
            if next_brace == -1: break
            try:
                obj, new_idx = decoder.raw_decode(content, next_brace)
                self.data.append(obj)
                idx = new_idx
            except json.JSONDecodeError:
                idx = next_brace + 1

        print(f"Loaded {len(self.data)} qs.")
        self.image_dir = Path(image_dir)
        if len(self.data) == 0:
            raise ValueError("No data")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        q = self.data[idx]
        image_path = self.image_dir / q['image_path']
        image = Image.open(image_path).convert('RGB')
        true_answer = str(q['true_answer'])
        q_type = "count" if true_answer.isdigit() else "other"

        return {
            "image": image,
            "question": q['question'],
            "true_answer": true_answer,
            "target_answer": str(q['target_answer']),
            "q_type": q_type
        }

In [ ]:
def pil_collate_fn(batch):
    return {key: [item[key] for item in batch] for key in batch[0].keys()}

### 3. Identifying the CLEVR Token IDs

In [ ]:
valid_clevr_answers = [str(i) for i in range(10)]

allowed_token_ids = []
token_to_string_map = {}

print("Mapping CLEVR answers to Gemma Token IDs:")
for ans in valid_clevr_answers:
    token_ids = proc.tokenizer.encode(ans, add_special_tokens=False)

    assert len(token_ids) == 1, f"Error: '{ans}' split into multiple tokens!"
    token_id = token_ids[0]

    allowed_token_ids.append(token_id)
    token_to_string_map[token_id] = ans

    print(f"Answer: '{ans}' -> Token ID: {token_id}")

## CAA vect application

### hook

In [ ]:
class SteeringHook:
    def __init__(self, model, steering_vector, target_layer, coefficient):
        self.model = model
        self.steering_vector = steering_vector
        self.target_layer = target_layer
        self.coefficient = coefficient
        self.handle = None

    def _hook_fn(self, module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output

        sv = self.steering_vector.to(hidden_states.device, dtype=hidden_states.dtype)
        if sv.dim() == 1:
            sv = sv.view(1, 1, -1)

        hidden_states.add_(self.coefficient * sv)

    def attach(self):
        layer_module = self.model.model.language_model.layers[self.target_layer]
        self.handle = layer_module.register_forward_hook(self._hook_fn)

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

### injection and evaluation

In [ ]:
def get_masked_clevr_prediction(logits, allowed_token_ids, token_to_string_map):
    vocab_size = logits.shape[-1]

    mask = torch.full(
        (1, vocab_size),
        float('-inf'),
        device=logits.device,
        dtype=logits.dtype
    )

    allowed_indices = torch.tensor(allowed_token_ids, device=logits.device)
    mask[0, allowed_indices] = logits[0, allowed_indices]

    probabilities = F.softmax(mask.to(torch.float32), dim=-1)
    predicted_token_id = torch.argmax(probabilities, dim=-1).item()
    prob_dict = {
        token_to_string_map[tid]: probabilities[0, tid].item()
        for tid in allowed_token_ids
    }

    return token_to_string_map[predicted_token_id], prob_dict

In [ ]:
def evaluate_steering_vector(model, processor, dataloader, steering_vector, target_layer, multiplier, allowed_token_ids, token_map):
    hook = SteeringHook(model, steering_vector, target_layer, multiplier)
    hook.attach()
    model.eval()

    metrics = defaultdict(lambda: {"total": 0, "correct_base": 0, "correct_shifted": 0, "p_true": 0.0, "p_target": 0.0})

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval multiplier: {multiplier}"):
            q = batch["question"][0]
            image = batch["image"][0]
            true_ans = batch["true_answer"][0]
            target_ans = batch["target_answer"][0]
            q_type = batch["q_type"][0]

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": q}
                    ]
                }
            ]

            eval_prompt = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = processor(text=eval_prompt, images=image, return_tensors="pt").to(model.device)
            outputs = model(**inputs)
            next_token_logits = outputs.logits[:, -1, :]
            pred_ans, prob_dict = get_masked_clevr_prediction(next_token_logits, allowed_token_ids, token_map)

            stats = metrics[q_type]
            stats["total"] += 1
            stats["p_true"] += prob_dict.get(true_ans, 0.0)
            stats["p_target"] += prob_dict.get(target_ans, 0.0)
            if pred_ans == true_ans:
                stats["correct_base"] += 1
            if pred_ans == target_ans:
                stats["correct_shifted"] += 1

    hook.remove()
    return metrics

### application and results

In [ ]:
ds_dir_fn='/content/drive/MyDrive/clevr/clevr_caa_extract_val'
vect_dir_fn = "/content/drive/MyDrive/clevr/clevr_overcount_plus1_layer"
layers = [15, 14, 13]
multipliers = [0.0, 0.5, 1.0, 1.5, 2.0]

In [ ]:
def eval_ds_steered(ds='100', layers=layers, ds_dir_fn=ds_dir_fn, vect_dir_fn=vect_dir_fn, multipliers=multipliers):
    extract_dataset = CLEVRCAAEvaluationDataset(f'{ds_dir_fn}{ds}.jsonl', path)
    dl = DataLoader(extract_dataset, batch_size=1, shuffle=False, collate_fn=pil_collate_fn)
    for target_layer in layers:
        print("Evaluating on layer", target_layer)
        steering_vector = torch.load(f"{vect_dir_fn}{target_layer}{ds}.pt")
        sweep_results = {}
        for mult in multipliers:
            print("Evaluating with", mult, 'multiplier')
            metrics = evaluate_steering_vector(
                model=mod,
                processor=proc,
                dataloader=dl,
                steering_vector=steering_vector,
                target_layer=target_layer,
                multiplier=mult,
                allowed_token_ids=allowed_token_ids,
                token_map=token_to_string_map
            )
            sweep_results[mult] = metrics
            for q_type, stats in metrics.items():
                total = stats["total"]
                if total == 0: continue
                base_acc = (stats["correct_base"] / total) * 100
                shift_acc = (stats["correct_shifted"] / total) * 100
                print(f"Type: {q_type} | Base Acc: {base_acc:5.1f}% | Shifted Acc: {shift_acc:5.1f}% | Avg P(Target): {stats['p_target']:5.3f}")
        return sweep_results

In [ ]:
eval_ds_steered('10')

In [ ]:
eval_ds_steered('100')

In [ ]:
eval_ds_steered('1k')